# Step-by-step implementation
The following are the steps to implement multi-query:
  1. Import necessary modules
  2. Set up the LangSmith and OpenAI API keys
  3. Prepare data and split text
  4. Index documents
  5. Generate multi-perspective query with LLM
  6. Retrieve documents using multi-query
  7. Run the RAG model


## 1. Import necessary modules

In [1]:
import os
import bs4
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from helpers import get_experientiallabs_llm
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

C:\Users\soura\AppData\Local\Temp\ipykernel_36828\1267605594.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


2026/09/09 13:58:38 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at D:\AI ENGINEERING\03_LangGraph_Fundamentals\.venv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


## 2. Set up the LangSmith and OpenAI API keys

In [2]:
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [3]:
# os.environ['LANGCHAIN_TRACING_V2'] = 'true'
# os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
# os.environ['LANGCHAIN_API_KEY'] = '' # Add your LangSmith LangChain API key
os.environ['LANGSMITH_PROJECT']='Multi-Query'

## 3. Prepare data and split text

In [4]:
loaders = [
    TextLoader("../../shared_data/blog.langchain.dev_announcing-langsmith_.txt", encoding="utf-8"),
    TextLoader("../../shared_data/blog.langchain.dev_automating-web-research_.txt", encoding="utf-8"),
]

docs = []
for loader in loaders:
    docs.extend(loader.load())

In [5]:
docs

[Document(metadata={'source': '../../shared_data/blog.langchain.dev_announcing-langsmith_.txt'}, page_content='URL: https://blog.langchain.dev/announcing-langsmith/\nTitle: Announcing LangSmith, a unified platform for debugging, testing, evaluating, and monitoring your LLM applications\n\nLangChain exists to make it as easy as possible to develop LLM-powered applications.\n\nWe started with an open-source Python package when the main blocker for building LLM-powered applications was getting a simple prototype working. We remember seeing Nat Friedman tweet in late 2022 that there was “not enough tinkering happening.” The LangChain open-source packages are aimed at addressing this and we see lots of tinkering happening now (Nat agrees)–people are building everything from chatbots over internal company documents to an AI dungeon master for a Dungeons and Dragons game.\n\nThe blocker has now changed. While it’s easy to build a prototype of an application in ~5 lines of LangChain code, it’s

In [6]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=400, chunk_overlap=60)
splits = text_splitter.split_documents(docs)

## 4. Index documents

In [7]:
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())

retriever = vectorstore.as_retriever()

## 5. Generate multi-perspective query with LLM

In [8]:
# Multi Query: Different Perspectives
template = """You are an AI language model assistant tasked with generating informative queries for a vector search engine.
The user has a question: "{question}"
Your goal/task is to create three variations of this question that capture different aspects of the user's intent. These variations will help the search engine retrieve relevant documents even if they don't use the exact keywords as the original question.
Provide these alternative questions, each on a new line.**
Original question: {question}"""

prompt_perspectives = ChatPromptTemplate.from_template(template)

generate_queries = (
    prompt_perspectives
    | get_experientiallabs_llm()
    | StrOutputParser()
    | (lambda x: x.split("\n"))
)

## 6. Retrieve documents using multi-query

In [9]:
def get_unique_union(documents: list[list]):
  """ Unique union of retrieved docs """
  # Flatten list of lists
  flattened_docs = [doc for sublist in documents for doc in sublist]

  # Option 1: Check library documentation for hashable attribute (e.g., 'id')
  if hasattr(flattened_docs[0], 'id'):  # Replace 'id' with the appropriate attribute
      unique_docs = list(set(doc.id for doc in flattened_docs))

  # Option 2: Convert to string (if suitable)
  else:
      unique_docs = list(set(str(doc) for doc in flattened_docs))

  return unique_docs

In [10]:
# Retrieve
question = "What is LangSmith, and why do we need it?"
retrieval_chain = generate_queries | retriever.map() | get_unique_union
docs = retrieval_chain.invoke({"question":question})
len(docs)

1

## 7. Run the RAG model

In [11]:
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

llm = get_experientiallabs_llm()

final_rag_chain = (
    {"context": retrieval_chain,
     "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})

'**LangSmith** is a platform from the LangChain team for developing, evaluating, debugging, and monitoring applications powered by large language models (LLMs).\n\nIt provides tools to:\n\n- **Trace and inspect** LLM calls, prompts, tool use, chains, and agent steps\n- **Debug failures** and understand why an application produced a particular response\n- **Evaluate quality** using datasets, human feedback, and automated evaluators\n- **Track performance**, including latency, token usage, and cost\n- **Manage prompt and application versions**\n- **Monitor production behavior** and identify regressions or recurring errors\n\nWe need it because LLM applications are often nondeterministic and involve multiple hidden steps. A normal application log may only show the final answer, while LangSmith can show the entire execution process. This makes it easier to improve reliability, control costs, compare changes, and maintain quality as an application moves from experimentation into production.